<a href="https://colab.research.google.com/github/ATKander/voc-sim-2026/blob/main/voc-sim-2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
pd.read_csv("metrics_base.csv").tail(1)


,year,pop,wage_index,wage_share,voc_price,state_rate,gini,cum_div,cash_mln,debt_mln,ships,unrest,regent_share
75,1660,197.9,175.8,53.2,368.5,5.0,0.796,413.0,2.31,0.0,16,0.0,46.5


In [2]:
%%writefile monte_carlo.py
"""Monte Carlo по калибровке движка VOC.
150 наборов параметров x 4 сида x 7 сценариев, парные сравнения.
Возобновление: прогресс пишется в mc_raw.csv после каждого draw.
"""
import os
import numpy as np
import pandas as pd
from voc_engine_v3 import BASE, MODERATE, STINGY, WARLIKE, run_dutch

N_DRAWS = 150
SEEDS = [42, 7, 2024, 31337]
CSV = "mc_raw.csv"

SCENARIOS = {
    "base":                      dict(MODERATE),
    "stingy":                    dict(STINGY),
    "warlike":                   dict(WARLIKE),
    "cf1_no_dispossession":      dict(MODERATE, disposs_on=False),
    "cf2_no_centralization":     dict(MODERATE, centralization_on=False),
    "cf3_no_permanent_capital":  dict(MODERATE, permanent_capital=False),
    "cf4_no_wisselbank":         dict(MODERATE, wisselbank=False),
}

# Только калибровочные параметры (div_rate — параметр политики, не трогаем).
SAMPLING = {
    "markup":       (2.2, 4.2),
    "ship_cost":    (0.07e6, 0.14e6),
    "silver":       (0.09e6, 0.16e6),
    "payroll_base": (0.20e6, 0.42e6),
    "infra_growth": (0.08e6, 0.18e6),
    "fail_prob":    (0.015, 0.05),
    "centr":        (0.03, 0.10),
    "dis_k":        (0.12, 0.30),
    "cong":         (0.04, 0.09),
}
OUT_COLS = ["pop", "wage_index", "voc_price", "gini", "cum_div",
            "cash_mln", "debt_mln", "regent_share", "unrest"]


def summarize(df):
    r = df.iloc[-1]                      # итог 1660 года
    return {c: r[c] for c in OUT_COLS}


def main():
    rng = np.random.default_rng(2026)
    rows = []
    done = set()
    if os.path.exists(CSV):
        prev = pd.read_csv(CSV)
        if len(prev):
            rows = prev.to_dict("records")
            done = set(zip(prev["draw"], prev["scenario"], prev["seed"]))
        print(f"продолжаю: {len(rows)} прогонов уже на диске")

    for d in range(N_DRAWS):
        draw = {k: rng.uniform(*v) for k, v in SAMPLING.items()}
        for name, pol in SCENARIOS.items():
            for s in SEEDS:
                if (d, name, s) in done:
                    continue
                cfg = dict(BASE, **pol, **draw)
                rec = summarize(run_dutch(cfg, seed=s))
                rec.update(draw=d, scenario=name, seed=s, **draw)
                rows.append(rec)
        pd.DataFrame(rows).to_csv(CSV, index=False)   # контрольная точка
    print(f"готово: {len(rows)} прогонов -> {CSV}")


if __name__ == "__main__":
    main()


Writing monte_carlo.py


In [3]:
!python monte_carlo.py


готово: 4200 прогонов -> mc_raw.csv


In [4]:
import pandas as pd

mc = pd.read_csv("mc_raw.csv")
piv = mc.pivot_table(index=["draw", "seed"], columns="scenario",
                     values=["pop", "gini", "regent_share", "cum_div", "voc_price"])
print(piv.groupby(level=1).median().round(3))

diff = piv[("pop", "cf1_no_dispossession")] - piv[("pop", "base")]
print(f"КФ-1 минус база: медиана {diff.median():.1f} тыс., P(ниже базы) = {(diff < 0).mean():.0%}")

g = piv[("gini", "cf2_no_centralization")] - piv[("gini", "base")]
print(f"КФ-2 минус база (Джини): медиана {g.median():.3f}, P(ниже на >0.1) = {(g < -0.1).mean():.0%}")

crash = mc[mc["scenario"] == "base"].groupby(["draw", "seed"])["voc_price"].last()
print(f"Краховых миров: {(crash < 100).mean():.0%}")


         cum_div                                             \
scenario    base cf1_no_dispossession cf2_no_centralization   
seed                                                          
7          413.0                413.0                 413.0   
42         413.0                413.0                 413.0   
2024       413.0                413.0                 413.0   
31337      413.0                413.0                 413.0   

                                                                     gini  \
scenario cf3_no_permanent_capital cf4_no_wisselbank stingy warlike   base   
seed                                                                        
7                             0.0             413.0  272.6   413.0  0.781   
42                            0.0             413.0  272.6   413.0  0.844   
2024                          0.0             413.0  272.6   413.0  0.856   
31337                         0.0             413.0  272.6   413.0  0.783   

                  

In [5]:
!apt-get install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen3:8b


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [6]:
!ollama list


NAME        ID              SIZE      MODIFIED      
qwen3:8b    500a1f067a9f    5.2 GB    3 seconds ago    


In [7]:
import requests, json, statistics

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

# Исторический состав Heeren XVII: Амстердам 8 голосов, Зеландия 4,
# Делфт 2, Роттердам 1, Хоорн 1, Энкхёйзен 1 — всего 17.
HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

PROMPT = """Year: {year}. VOC treasury: {cash_mln:.2f}M guilders. Debt: {debt_mln:.2f}M.
Share price: {price:.0f}. Recent dividends: {recent_divs}. Popular unrest: {unrest:.2f}.
War with Spain/Portugal: {war}.
Decide: div_rate (0-20, % of capital), form ("cash" or "spices"),
war (true/false), ships (1-20, new ships per year).
Merchants historically paid LOW dividends in spices to keep capital compounding.
Answer in JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}"""


def ask_one(city, persona, temp, ctx):
    sys = SYSTEM.format(city=city, year=ctx["year"])
    usr = PROMPT.format(**ctx)
    try:
        r = requests.post(URL, json={
            "model": MODEL, "format": "json",
            "messages": [{"role": "system", "content": sys},
                         {"role": "user", "content": usr + " /no_think"}],
            "options": {"temperature": temp}, "stream": False}, timeout=120)
        v = json.loads(r.json()["message"]["content"])
        return {"div_rate": float(v.get("div_rate", 7)),
                "form": str(v.get("form", "cash"))[:5],
                "war": bool(v.get("war", False)),
                "ships": max(1, min(20, int(v.get("ships", 16))))}
    except Exception:
        return {"div_rate": 7.0, "form": "cash", "war": False, "ships": 16}


def assemble(ctx):
    votes = [ask_one(c, p, t, ctx) for c, p, t in HEEREN]
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > 8 else "cash",
            "war": sum(v["war"] for v in votes) > 8,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


In [8]:
!nvidia-smi


Sat Nov 14 13:37:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
print("Разброс голосов:", sorted(v["div_rate"] for v in decision["votes"]))


KeyboardInterrupt: 

In [10]:
import time
t0 = time.time()
r = !curl -s --max-time 120 http://localhost:11434/api/generate -d '{{"model": "qwen3:8b", "prompt": "say hi", "stream": false, "think": false}}'
print(f"время: {time.time()-t0:.1f} с")
print(r[:1][0][:300] if r else "ПУСТОЙ ОТВЕТ")


время: 0.1 с
ПУСТОЙ ОТВЕТ


In [11]:
!pkill ollama
import subprocess, time
subprocess.Popen(["ollama", "serve"],
                 stdout=open("ollama_serve.log", "w"),
                 stderr=subprocess.STDOUT)
time.sleep(6)
!head -50 ollama_serve.log


time=2026-11-14T13:43:16.406Z level=INFO source=routes.go:2474 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

In [12]:
!ollama pull qwen3:8b


In [13]:
import time
t0 = time.time()
r = !curl -s --max-time 180 http://localhost:11434/api/generate -d '{{"model": "qwen3:8b", "prompt": "say hi", "stream": false, "think": false}}'
print(f"время: {time.time()-t0:.1f} с")
print((r[0] if r else "")[:300] if r else "ПУСТОЙ ОТВЕТ")


время: 4.1 с
{"model":"qwen3:8b","created_at":"2026-11-14T13:44:57.958442903Z","response":"Hello! How can I assist you today? 😊","done":true,"done_reason":"stop","context":[151644,872,198,36790,15588,608,2152,5854,766,151645,198,151644,77091,198,151667,271,151668,271,9707,0,2585,646,358,7789,498,3351,30,26525,23


In [14]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
print("Разброс голосов:", sorted(v["div_rate"] for v in decision["votes"]))


Медианы: {'div_rate': 7.0, 'form': 'cash', 'war': False, 'max_launch': 16}
Разброс голосов: [0.0, 0.0, 0.0, 0.0, 0.0, 5.0, 5.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0]


In [15]:
for v in decision["votes"]:
    print(v)


{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 0.0, 'form': 'spice', 'war': True, 'ships': 5}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 0.0, 'form': 'spice', 'war': True, 'ships': 10}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 5.0, 'form': 'spice', 'war': True, 'ships': 5}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 0.0, 'form': 'spice', 'war': True, 'ships': 5}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 7.0, 'form': 'cash', 'war': False, 'ships': 16}
{'div_rate': 5.0, 'form': 'spice', 'war': True, 'ships': 5}
{'div_rate': 0.0, 'form': 'spice', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 's

In [17]:
import time, requests
sys_text = SYSTEM.format(city="Амстердам", year=1602)
usr_text = PROMPT.format(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
                         recent_divs="нет", unrest=0.0, war=True)
t0 = time.time()
r = requests.post("http://localhost:11434/api/chat", json={
    "model": "qwen3:8b", "think": False,
    "messages": [{"role": "system", "content": sys_text},
                 {"role": "user", "content": usr_text}],
    "options": {"temperature": 0.7}, "stream": False}, timeout=120)
print(f"время: {time.time()-t0:.1f} с")
print(repr(r.json()["message"]["content"][:600]))


время: 1.3 с
'{"div_rate": 5, "form": "spices", "war": true, "ships": 10}'


In [18]:
def ask_one(city, year, ctx, attempts=3):
    sys_text = SYSTEM.format(city=city, year=year)
    usr_text = PROMPT.format(year=year, cash_mln=ctx["cash_mln"], debt_mln=ctx["debt_mln"],
                             price=ctx["price"],
                             recent_divs="нет" if not ctx["recent_divs"]
                             else ", ".join(f"{d}%" for d in ctx["recent_divs"]),
                             unrest=ctx["unrest"], war=bool(ctx["war"]))
    for i in range(attempts):
        try:
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "qwen3:8b", "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr_text}],
                "options": {"temperature": 0.7 if i == 0 else 0.3}, "stream": False},
                timeout=120)
            raw = r.json()["message"]["content"]
            m = re.search(r"\{.*\}", raw, re.S)
            v = json.loads(m.group(0))
            form = v["form"]
            if form in ("spices", "spice"): form = "spice"      # ключевой фикс
            elif form not in ("cash",): form = "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception:
            continue
    return None


In [19]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
for v in decision["votes"]:
    print(v)


TypeError: 'float' object is not subscriptable

In [20]:
print(repr(PROMPT))


'Year: {year}. VOC treasury: {cash_mln:.2f}M guilders. Debt: {debt_mln:.2f}M.\nShare price: {price:.0f}. Recent dividends: {recent_divs}. Popular unrest: {unrest:.2f}.\nWar with Spain/Portugal: {war}.\nDecide: div_rate (0-20, % of capital), form ("cash" or "spices"),\nwar (true/false), ships (1-20, new ships per year).\nMerchants historically paid LOW dividends in spices to keep capital compounding.\nAnswer in JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}'


In [21]:
def ask_one(city, year, ctx, attempts=3):
    sys_text = SYSTEM.format(city=city, year=year)
    divs = "нет" if not ctx["recent_divs"] else ", ".join(f"{d}%" for d in ctx["recent_divs"])
    usr_text = (
        f"Year: {year}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\n"
        f"Share price: {ctx['price']:.0f}. Recent dividends: {divs}. "
        f"Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spice"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Merchants historically paid LOW dividends in spices to keep capital compounding.\n"
        'Answer in JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}'
    )
    for i in range(attempts):
        try:
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "qwen3:8b", "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr_text}],
                "options": {"temperature": 0.7 if i == 0 else 0.3}, "stream": False},
                timeout=120)
            raw = r.json()["message"]["content"]
            m = re.search(r"\{.*\}", raw, re.S)
            v = json.loads(m.group(0))
            form = v["form"]
            if form in ("spices", "spice"): form = "spice"
            elif form != "cash": form = "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception:
            continue
    return None


In [23]:
def ask_one(city, year, ctx, attempts=3):
    sys_text = SYSTEM.format(city=city, year=year)
    rd = ctx["recent_divs"]
    if rd is None or rd == "":
        divs = "нет"
    elif isinstance(rd, (list, tuple)):
        divs = "нет" if len(rd) == 0 else ", ".join(f"{d}%" for d in rd)
    else:
        divs = f"{rd}%"
    usr_text = (
        f"Year: {year}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\n"
        f"Share price: {ctx['price']:.0f}. Recent dividends: {divs}. "
        f"Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spice"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Merchants historically paid LOW dividends in spices to keep capital compounding.\n"
        'Answer in JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}'
    )
    for i in range(attempts):
        try:
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "qwen3:8b", "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr_text}],
                "options": {"temperature": 0.7 if i == 0 else 0.3}, "stream": False},
                timeout=120)
            raw = r.json()["message"]["content"]
            m = re.search(r"\{.*\}", raw, re.S)
            v = json.loads(m.group(0))
            form = v["form"]
            if form in ("spices", "spice"): form = "spice"
            elif form != "cash": form = "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception:
            continue
    return None


In [24]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
for v in decision["votes"]:
    print(v)


TypeError: 'float' object is not subscriptable

In [25]:
!grep -n -A 45 "def assemble" voc_engine_v3.py
!grep -n -A 45 "def ask_one" voc_engine_v3.py


In [26]:
print(repr(SYSTEM))


'You are a bewindhebber (director) of the VOC, an Amsterdam merchant of the chamber of {city}, year {year}. Historical context: VOC has permanent capital of 6.42M guilders, trade brings 100-300% on voyages. Decide the annual policy. Respond ONLY with JSON.'


In [27]:
import inspect
print(inspect.getsource(assemble))


def assemble(ctx):
    votes = [ask_one(c, p, t, ctx) for c, p, t in HEEREN]
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > 8 else "cash",
            "war": sum(v["war"] for v in votes) > 8,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}



In [28]:
src = open("voc_engine_v3.py").read()
i = src.find("def ask_one")
print(src[i:i+1600] if i != -1 else "ask_one в файле не найден")
print("---")
print("HEEREN[:3]:", HEEREN[:3])
print("---")
print(SYSTEM)


ask_one в файле не найден
---
HEEREN[:3]: [('Amsterdam', 'merchant-am0', 0.5), ('Amsterdam', 'merchant-am1', 0.6), ('Amsterdam', 'merchant-am2', 0.7)]
---
You are a bewindhebber (director) of the VOC, an Amsterdam merchant of the chamber of {city}, year {year}. Historical context: VOC has permanent capital of 6.42M guilders, trade brings 100-300% on voyages. Decide the annual policy. Respond ONLY with JSON.


In [29]:
def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\nShare price: {ctx['price']:.0f}. "
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Merchants historically paid LOW dividends in spices to keep capital compounding.\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "qwen3:8b", "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False},
                timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception:
            continue
    return None


In [30]:
def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > 8 else "cash",
            "war": sum(v["war"] for v in votes) > 8,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


In [31]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
for v in decision["votes"]:
    print(v)


StatisticsError: no median for empty data

In [32]:
import re, json, requests, statistics


In [33]:
def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\nShare price: {ctx['price']:.0f}. "
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Merchants historically paid LOW dividends in spices to keep capital compounding.\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post("http://localhost:11434/api/chat", json={
                "model": "qwen3:8b", "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False},
                timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))   # видимость вместо тишины
            continue
    return None


In [34]:
ctx = dict(year=1602, cash_mln=6.42, debt_mln=0.0, price=100.0,
           recent_divs=[], unrest=0.0, war=True)
decision = assemble(ctx)
print("Медианы:", {k: v for k, v in decision.items() if k != "votes"})
for v in decision["votes"]:
    print(v)


Медианы: {'div_rate': 0.0, 'form': 'spices', 'war': True, 'max_launch': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 5.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 5.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 5.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 5.0, 'form': 'spices', 'war': True, 'ships': 10}
{'div_rate': 0.0, 'form': 'spices', 'war': True, 'ships'

In [35]:
import inspect
print(inspect.getsource(run_dutch))


NameError: name 'run_dutch' is not defined

In [36]:
src = open("voc_engine_v3.py").read()
i = src.find("def run_dutch")
print(src[i:i+4000] if i != -1 else "run_dutch не найден — покажу список функций")
if i == -1:
    for n, line in enumerate(src.splitlines(), 1):
        if line.startswith("def ") or line.startswith("class "):
            print(n, line)


def run_dutch(p, years=range(1585, 1661), seed=42):
    rng = np.random.default_rng(seed)
    pop, W = 30000.0, 105.0
    wage_share = 0.35; state_rate = 8.33; prev_g = 0.03
    voc = False; nominal = 0.0; cash = 0.0; debt = 0.0; infra = 0.0
    cum_div = 0.0; voc_price = np.nan; unrest = 0.0
    fleet_out, fleet_mid, fleet_ret = [], [], []
    pool = 0.0; pop1600 = pop
    caps = rng.lognormal(6.5, p['sigma'], p['N'])
    caps = caps / caps.sum() * 3.67e6
    regent_idx = np.argsort(caps)[::-1][:17]
    records = []

    for year in years:
        ret = 0.060 if year <= 1620 else (0.045 if year <= 1650 else 0.032)
        if p['wisselbank'] and year >= 1609: ret += 0.004
        if year in WAR_YEARS: ret -= 0.010

        if year == 1602:
            voc = True
            scale = (pop1600 / POP_REF) if p.get('pop_scale_nominal') else 1.0
            nominal = 6.42e6 * scale
            cash = nominal; pool = nominal

        jobs = float(np.clip(0.006 + 0.55*ret, 0.004, 0.028))
     

In [37]:
src = open("voc_engine_v3.py").read()
i = src.find("def run_dutch")
print(src[i+4000:i+9000])


_rate']/100 * nominal)
                    cash -= amount
                    eff = amount * (0.77 if p['div_form'] == "spices" else 1.0)
                    caps *= (1 + eff / caps.sum()); cum_div += eff / nominal * 100
                    unrest *= 0.5 if p['div_form'] == "cash" else 0.9
                else:
                    unrest += 0.2
            else:
                if pool < 0.5*nominal/21: unrest += 0.1
            if p['centralization_on']:
                mid_m = (caps >= 1000) & (caps < 20000)
                fail = (rng.random(len(caps)) < p['fail_prob']) & mid_m
                if fail.any():
                    caps[int(np.argmax(caps))] += caps[fail].sum() * p['fail_recover']
                    caps[fail] = 0.0
                f = p['centr'] + p['centr_unrest'] * min(unrest, 3)
                if 1636 <= year <= 1639: f += 0.20
                small = caps < 1000
                if small.any():
                    lot = caps[small].sum() * f; caps[small] *= (1 - f

In [38]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

# Исторический состав Heeren XVII: Амстердам 8, Зеландия 4,
# Делфт 2, Роттердам 1, Хоорн 1, Энкхёйзен 1.
HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\nShare price: {ctx['price']:.0f}. "
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Merchants historically paid LOW dividends in spices to keep capital compounding.\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Writing voc_council.py


In [39]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

# 1) импорт совета
sub("def run_dutch(p,", "import voc_council\n\n\ndef run_dutch(p,")

# 2) память совета перед циклом лет
sub("    regent_idx = np.argsort(caps)[::-1][:17]\n    records = []",
    "    regent_idx = np.argsort(caps)[::-1][:17]\n    records = []\n"
    "    war_last = False; div_history = []\n"
    "    council_pol = dict(div_rate=0.0, form=\"spices\", war=False, max_launch=10)")

# 3) собрание раз в 5 лет; решения перекрывают конфиг
sub("            war = p['war'] and year >= 1605",
    '''            if year == 1602 or (year - 1602) % 5 == 0:
                ctx = dict(year=year, cash_mln=cash/1e6, debt_mln=debt/1e6,
                           price=100.0 if np.isnan(voc_price) else float(voc_price),
                           recent_divs=div_history[-3:], unrest=unrest, war=war_last)
                council_pol = voc_council.assemble(ctx)
                if council_pol["div_rate"] > 0:
                    div_history.append(council_pol["div_rate"])
                war_last = council_pol["war"]
            war = council_pol["war"] and year >= 1605''')

# 4) спуск кораблей
sub("while ships_launched < p['max_launch']",
    'while ships_launched < int(council_pol["max_launch"])')

# 5) дивиденды: условие, ставка, форма
sub("if p['div_rate'] > 0 and cash > p['div_reserve']:",
    'if council_pol["div_rate"] > 0 and cash > p[\'div_reserve\']:')
sub("p['div_rate']/100 * nominal", 'council_pol["div_rate"]/100 * nominal')
sub("p['div_form'] == \"spices\"", 'council_pol["form"] == "spices"')

open("voc_engine_v3.py", "w").write(src)
print("патч применён")


патч применён


In [40]:
import importlib, voc_engine_v3
importlib.reload(voc_engine_v3)
from voc_engine_v3 import BASE, MODERATE

df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)
df[df["year"] >= 1600].tail(15)


,year,pop,wage_index,wage_share,voc_price,state_rate,gini,cum_div,cash_mln,debt_mln,ships,unrest,regent_share
61,1646,179.7,176.6,51.1,10.0,5.0,0.923,0.0,1.23,0.00,5,9.0,69.0
62,1647,182.1,177.7,51.2,10.0,5.0,0.927,0.0,1.12,0.00,5,9.2,69.8
63,1648,184.2,178.8,51.4,10.0,5.0,0.929,0.0,1.01,0.00,5,9.4,70.3
64,1649,186.3,179.9,51.5,10.0,5.0,0.929,0.0,0.89,0.00,5,9.6,70.3
65,1650,188.2,181.2,51.7,10.0,5.0,0.931,0.0,0.75,0.00,5,9.8,70.7
66,1651,189.6,180.5,51.8,10.0,5.0,0.932,0.0,0.62,0.00,5,10.0,71.2
67,1652,190.0,179.6,52.0,10.0,5.0,0.934,0.0,0.00,0.30,10,10.2,71.5
68,1653,190.4,179.0,52.1,10.0,5.0,0.934,0.0,0.00,0.43,6,10.4,71.6
69,1654,190.8,178.4,52.3,10.0,5.0,0.935,0.0,0.00,0.55,4,10.6,71.9
70,1655,192.2,178.2,52.4,10.0,5.0,0.937,0.0,0.01,0.52,2,10.8,72.5


In [41]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M.\nShare price: {ctx['price']:.0f}. "
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}.\n"
        f"War with Spain/Portugal: {bool(ctx['war'])}.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"),\n'
        "war (true/false), ships (1-20, new ships per year).\n"
        "Historical practice: dividends are LOW but OCCASIONAL - the VOC paid its "
        "first dividend in 1610 (in spices), then irregularly. Skipping dividends "
        "for many years in a row angers shareholders and feeds popular unrest: "
        "if unrest exceeds ~2, shareholders expect a payout (cash dividends calm "
        "the crowds far better than spice payments). Never vote for new ships if "
        "the treasury is empty or debt exceeds 1M guilders.\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [42]:
import voc_council, importlib
importlib.reload(voc_council)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes"), {k: v for k, v in d.items() if k != "votes"}
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{d2["war"]},{d2["max_launch"]}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged


In [43]:
import importlib, voc_engine_v3
importlib.reload(voc_engine_v3)
from voc_engine_v3 import BASE, MODERATE

open("council_log.csv", "w").write("year,div_rate,form,war,max_launch\n")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
df[df["year"] >= 1600].tail(15)


year,div_rate,form,war,max_launch
1602,0.0,spices,False,2
1607,0.0,spices,False,2
1612,5.0,cash,False,3
1617,5.0,cash,False,2
1622,5.0,cash,False,2
1627,5.0,cash,False,1
1632,5.0,cash,False,1
1637,5.0,cash,False,1
1642,5.0,cash,False,1
1647,5.0,cash,False,1
1652,5.0,cash,False,1
1657,5.0,cash,False,1



,year,pop,wage_index,wage_share,voc_price,state_rate,gini,cum_div,cash_mln,debt_mln,ships,unrest,regent_share
61,1646,179.7,176.6,51.1,26.8,5.0,0.909,25.8,0.0,13.48,0,5.83,65.1
62,1647,182.1,177.7,51.2,26.8,5.0,0.912,25.8,0.0,14.13,0,6.03,65.8
63,1648,184.2,178.8,51.4,26.8,5.0,0.913,25.8,0.0,14.80,0,6.23,65.9
64,1649,186.3,179.9,51.5,26.8,5.0,0.914,25.8,0.0,15.47,0,6.43,66.2
65,1650,188.2,181.2,51.7,26.8,5.0,0.914,25.8,0.0,16.15,0,6.63,66.2
66,1651,189.6,180.5,51.8,26.8,5.0,0.915,25.8,0.0,16.84,0,6.83,66.2
67,1652,190.0,179.6,52.0,26.8,5.0,0.915,25.8,0.0,17.53,0,7.03,66.3
68,1653,190.4,179.0,52.1,26.8,5.0,0.916,25.8,0.0,18.23,0,7.23,66.4
69,1654,190.8,178.4,52.3,26.8,5.0,0.917,25.8,0.0,18.94,0,7.43,66.8
70,1655,192.2,178.2,52.4,26.8,5.0,0.919,25.8,0.0,19.65,0,7.63,67.1


In [44]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

sub("    war_last = False; div_history = []",
    "    war_last = True; div_history = []; prev_rev = 0.0")

sub("""            if year == 1602 or (year - 1602) % 5 == 0:
                ctx = dict(year=year, cash_mln=cash/1e6, debt_mln=debt/1e6,
                           price=100.0 if np.isnan(voc_price) else float(voc_price),
                           recent_divs=div_history[-3:], unrest=unrest, war=war_last)""",
    """            if year == 1602 or (year - 1602) % 5 == 0:
                ctx = dict(year=year, cash_mln=cash/1e6, debt_mln=debt/1e6,
                           price=100.0 if np.isnan(voc_price) else float(voc_price),
                           recent_divs=div_history[-3:], unrest=unrest, war=war_last,
                           fleet_sea=len(fleet_out) + len(fleet_mid) + len(fleet_ret),
                           last_rev_mln=prev_rev/1e6,
                           ships_last=records[-1]["ships"] if records else 0)""")

sub("        reg_share = 100*caps[regent_idx].sum()/caps.sum() if voc else np.nan",
    "        prev_rev = revenue\n"
    "        reg_share = 100*caps[regent_idx].sum()/caps.sum() if voc else np.nan")

open("voc_engine_v3.py", "w").write(src)
print("патч 2 применён")


патч 2 применён


In [45]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Debt: {ctx['debt_mln']:.2f}M. Share price: {ctx['price']:.0f}.\n"
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}. "
        f"War with Spain/Portugal ongoing: {bool(ctx['war'])}.\n"
        f"Fleet at sea: {ctx['fleet_sea']} ships. Ships launched last year: "
        f"{ctx['ships_last']}. Trade revenue last year: {ctx['last_rev_mln']:.2f}M guilders.\n"
        "Economics: each return voyage multiplies the cargo invested 2-4x; "
        "typical annual running costs are roughly 0.5-1.5M guilders. A fleet too "
        "small cannot cover costs and trade will shrink; expanding with an empty "
        "treasury creates debt. Occasional dividends keep shareholders calm; "
        "skipping them for years raises unrest.\n"
        "Your goal: the long-term prosperity of the Company and steady returns "
        "for its shareholders.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"), '
        'war (true/false: whether the VOC maintains its armed posture - convoys '
        "and privateering - this period), ships (1-20, new ships per year).\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [46]:
!cp metrics_llm_council.csv metrics_llm_council_v2_caution.csv
!cp council_log.csv council_log_v2_caution.csv

import importlib, voc_council, voc_engine_v3
importlib.reload(voc_council); importlib.reload(voc_engine_v3)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes"), {k: v for k, v in d.items() if k != "votes"}
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{d2["war"]},{d2["max_launch"]}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged

from voc_engine_v3 import BASE, MODERATE
open("council_log.csv", "w").write("year,div_rate,form,war,max_launch\n")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
df[df["year"] >= 1600].tail(10)


year,div_rate,form,war,max_launch
1602,5.0,cash,True,5
1607,5.0,cash,True,5
1612,5.0,cash,True,5
1617,5.0,cash,True,5
1622,5.0,cash,True,5
1627,5.0,cash,True,5
1632,5.0,cash,True,4
1637,5.0,cash,True,5
1642,5.0,spices,True,3
1647,5.0,spices,True,5
1652,5.0,cash,True,5
1657,5.0,cash,True,5



,year,pop,wage_index,wage_share,voc_price,state_rate,gini,cum_div,cash_mln,debt_mln,ships,unrest,regent_share
66,1651,189.6,180.5,51.8,83.5,5.0,0.843,113.1,0.0,10.85,0,4.65,52.3
67,1652,190.0,179.6,52.0,83.5,5.0,0.843,113.1,0.0,11.55,0,4.85,52.3
68,1653,190.4,179.0,52.1,83.5,5.0,0.843,113.1,0.0,12.25,0,5.05,52.3
69,1654,190.8,178.4,52.3,83.5,5.0,0.843,113.1,0.0,12.95,0,5.25,52.3
70,1655,192.2,178.2,52.4,83.5,5.0,0.846,113.1,0.0,13.66,0,5.45,52.7
71,1656,193.5,177.7,52.6,83.5,5.0,0.847,113.1,0.0,14.38,0,5.65,52.8
72,1657,194.7,177.2,52.7,83.5,5.0,0.849,113.1,0.0,15.11,0,5.85,53.2
73,1658,195.8,176.7,52.9,83.5,5.0,0.849,113.1,0.0,15.84,0,6.05,53.2
74,1659,196.9,176.3,53.0,83.5,5.0,0.850,113.1,0.0,16.57,0,6.25,53.3
75,1660,197.9,175.8,53.2,83.5,5.0,0.853,113.1,0.0,17.32,0,6.45,53.8


In [47]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

sub("                           last_rev_mln=prev_rev/1e6,",
    "                           last_rev_mln=prev_rev/1e6,\n"
    "                           cash_free_mln=max(0.0, (cash - p['div_reserve'])/1e6),\n"
    "                           ship_cost_mln=p['ship_cost']/1e6,")

open("voc_engine_v3.py", "w").write(src)
print("патч 3 применён")


патч 3 применён


In [48]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Free capital above reserve: {ctx['cash_free_mln']:.2f}M. "
        f"Debt: {ctx['debt_mln']:.2f}M. Share price: {ctx['price']:.0f}.\n"
        f"Recent dividends: {divs}. Popular unrest: {ctx['unrest']:.2f}. "
        f"War with Spain/Portugal ongoing: {bool(ctx['war'])}.\n"
        f"Fleet at sea: {ctx['fleet_sea']} ships. Ships launched last year: "
        f"{ctx['ships_last']}. Trade revenue last year: {ctx['last_rev_mln']:.2f}M.\n"
        f"Budget facts: one ship costs ~{ctx['ship_cost_mln']:.2f}M to build and outfit; "
        "dividends are paid only from free capital above the reserve; the Company "
        "carries fixed running costs every year even if no fleet is launched.\n"
        "Economics: each ship at sea invests its cargo and returns roughly 2-4x "
        "within a few years. A shrinking fleet means shrinking revenue - if the "
        "fleet is small, rebuilding it is the first priority, even at the price of "
        "paying no dividends for several years. Dividends are paid only when the "
        "treasury can afford them without new debt.\n"
        "Your goal: the long-term prosperity of the Company and steady returns "
        "for its shareholders.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"), '
        'war (true/false: whether the VOC maintains its armed posture - convoys '
        "and privateering - this period), ships (1-20, new ships per year, within "
        "the free capital budget).\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [49]:
!cp metrics_llm_council.csv metrics_llm_council_v3_moderate.csv
!cp council_log.csv council_log_v3_moderate.csv

import importlib, voc_council, voc_engine_v3
importlib.reload(voc_council); importlib.reload(voc_engine_v3)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes"), {k: v for k, v in d.items() if k != "votes"}
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{d2["war"]},{d2["max_launch"]}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged

from voc_engine_v3 import BASE, MODERATE
open("council_log.csv", "w").write("year,div_rate,form,war,max_launch\n")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
r = df[df["year"] == 1660].iloc[0]
print(f"1660: price={r['voc_price']} cum_div={r['cum_div']} debt={r['debt_mln']} "
      f"ships={r['ships']} unrest={r['unrest']} regents={r['regent_share']}")


year,div_rate,form,war,max_launch
1602,0.0,cash,True,10
1607,0.0,spices,True,10
1612,0.0,spices,True,10
1617,0.0,spices,True,10
1622,0.0,spices,True,10
1627,0.0,spices,True,10
1632,0.0,spices,True,10
1637,0.0,spices,True,10
1642,0.0,spices,True,10
1647,0.0,cash,True,10
1652,0.0,spices,True,10
1657,0.0,spices,True,10

1660: price=10.0 cum_div=0.0 debt=0.0 ships=10.0 unrest=11.8 regents=72.2


In [50]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

sub("    war_last = True; div_history = []; prev_rev = 0.0",
    "    war_last = True; div_history = []; prev_rev = 0.0; last_div_year = None")

sub("""                if council_pol["div_rate"] > 0:
                    div_history.append(council_pol["div_rate"])""",
    """                if council_pol["div_rate"] > 0:
                    div_history.append(council_pol["div_rate"])
                    last_div_year = year""")

sub("                           ship_cost_mln=p['ship_cost']/1e6,",
    "                           ship_cost_mln=p['ship_cost']/1e6,\n"
    "                           years_since_div=(year - last_div_year) if last_div_year else (year - 1602 + 1),\n"
    "                           credit_left_mln=max(0.0, (p['credit_limit'] - debt)/1e6),")

# корабли в кредит: бюджет расширяется на свободный лимит,
# отрицательный кассовый остаток движок сам конвертирует в долг
sub("                budget = cash - p['div_reserve']",
    "                budget = cash - p['div_reserve'] + max(0.0, p['credit_limit'] - debt)")

open("voc_engine_v3.py", "w").write(src)
print("патч 4 применён")


патч 4 применён


In [51]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Free capital above reserve: {ctx['cash_free_mln']:.2f}M. "
        f"Debt: {ctx['debt_mln']:.2f}M. Share price: {ctx['price']:.0f}.\n"
        f"Recent dividends: {divs}. Years since last dividend: {ctx['years_since_div']}. "
        f"Popular unrest: {ctx['unrest']:.2f}. "
        f"War with Spain/Portugal ongoing: {bool(ctx['war'])}.\n"
        f"Fleet at sea: {ctx['fleet_sea']} ships. Ships launched last year: "
        f"{ctx['ships_last']}. Trade revenue last year: {ctx['last_rev_mln']:.2f}M.\n"
        f"Credit: the Company may borrow for shipbuilding; {ctx['credit_left_mln']:.1f}M "
        f"of the credit line remains. One ship costs ~{ctx['ship_cost_mln']:.2f}M; "
        "borrowed money costs interest (~3-5% per year).\n"
        "Economics and politics: each ship at sea returns roughly 2-4x its cargo "
        "within a few years, so the fleet is the engine of all income. But unrest "
        "rises by ~0.2 per year while shareholders go unpaid, and once unrest "
        "exceeds 3 the share price collapses to its floor and riots threaten - "
        "restoring calm later costs far more than a timely dividend.\n"
        "Historical practice: the real VOC usually reinvested, but paid substantial "
        "dividends roughly every 5-10 years. A healthy policy keeps the fleet at "
        "strength (8-16 ships), stays out of unaffordable debt, and pays a dividend "
        "in cash or spices at least once every 5-10 years, especially when unrest "
        "is above 2. Rebuilding a shrunken fleet takes priority over dividends; "
        "once the fleet is strong, dividends resume.\n"
        'Decide: div_rate (0-20, % of capital), form ("cash" or "spices"), '
        'war (true/false: maintain the armed posture this period), '
        "ships (1-20, new ships per year, within free capital plus credit).\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [52]:
!cp metrics_llm_council.csv metrics_llm_council_v4_hoard.csv
!cp council_log.csv council_log_v4_hoard.csv

import importlib, voc_council, voc_engine_v3
importlib.reload(voc_council); importlib.reload(voc_engine_v3)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes"), {k: v for k, v in d.items() if k != "votes"}
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{d2["war"]},{d2["max_launch"]}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged

from voc_engine_v3 import BASE, MODERATE
open("council_log.csv", "w").write("year,div_rate,form,war,max_launch\n")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
print("разброс голосов 1647:", [v["div_rate"] for v in df.attrs.get("votes", [])])
r = df[df["year"] == 1660].iloc[0]
print(f"1660: price={r['voc_price']} cum_div={r['cum_div']} debt={r['debt_mln']} "
      f"ships={r['ships']} unrest={r['unrest']} regents={r['regent_share']}")


year,div_rate,form,war,max_launch
1602,0.0,spices,True,8
1607,5.0,cash,True,2
1612,5.0,cash,True,2
1617,5.0,cash,True,2
1622,0.0,spices,True,2
1627,0.0,spices,True,2
1632,0.0,spices,True,16
1637,0.0,spices,True,16
1642,0.0,spices,True,16
1647,0.0,spices,True,16
1652,0.0,spices,True,16
1657,0.0,spices,True,16

разброс голосов 1647: []
1660: price=52.5 cum_div=65.4 debt=22.84 ships=0.0 unrest=8.0 regents=61.7


In [53]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

sub("    war_last = True; div_history = []; prev_rev = 0.0; last_div_year = None",
    "    war_last = True; div_history = []; prev_rev = 0.0; last_div_year = None\n"
    "    prev_snap = None; last_mandate = None")

sub("            if year == 1602 or (year - 1602) % 5 == 0:\n                ctx = dict(year=year,",
    '''            if year == 1602 or (year - 1602) % 5 == 0:
                if prev_snap is None:
                    retro = ("This is the first meeting; there is no previous "
                             "mandate to review.")
                else:
                    fl_now = len(fleet_out) + len(fleet_mid) + len(fleet_ret)
                    retro = (f"Review of your last {year - prev_snap['at']}-year mandate "
                             f"({prev_snap['mandate']} ships/yr): fleet {prev_snap['fleet']}->{fl_now}, "
                             f"revenue {prev_snap['rev']/1e6:.2f}M->{prev_rev/1e6:.2f}M per year, "
                             f"debt {prev_snap['debt']/1e6:.2f}M->{debt/1e6:.2f}M, "
                             f"unrest {prev_snap['unrest']:.2f}->{unrest:.2f}, "
                             f"cumulative dividends {prev_snap['cum_div']:.1f}%->{cum_div:.1f}%.")
                ctx = dict(year=year,''')

sub("                           credit_left_mln=max(0.0, (p['credit_limit'] - debt)/1e6),",
    "                           credit_left_mln=max(0.0, (p['credit_limit'] - debt)/1e6),\n"
    "                           retro=retro,")

sub('                war_last = council_pol["war"]',
    '''                prev_snap = dict(fleet=len(fleet_out) + len(fleet_mid) + len(fleet_ret),
                                 rev=prev_rev, debt=debt, unrest=unrest,
                                 cum_div=cum_div, mandate=last_mandate, at=year)
                war_last = council_pol["war"]
                last_mandate = council_pol["max_launch"]''')

open("voc_engine_v3.py", "w").write(src)
print("патч 5 применён")


патч 5 применён


In [54]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"{ctx['retro']}\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Free capital above reserve: {ctx['cash_free_mln']:.2f}M. "
        f"Debt: {ctx['debt_mln']:.2f}M. Share price: {ctx['price']:.0f}.\n"
        f"Recent dividends: {divs}. Years since last dividend: {ctx['years_since_div']}. "
        f"Unrest: {ctx['unrest']:.2f}. War ongoing: {bool(ctx['war'])}.\n"
        f"Fleet at sea: {ctx['fleet_sea']}. Launched last year: {ctx['ships_last']}. "
        f"Revenue last year: {ctx['last_rev_mln']:.2f}M. "
        f"Credit remaining: {ctx['credit_left_mln']:.1f}M "
        f"(one ship ~{ctx['ship_cost_mln']:.2f}M, borrowed money costs 3-5%/yr).\n"
        "Steady-state guidance: a healthy VOC launches a STEADY 8-12 ships per "
        "year and pays a dividend every 5-10 years. Do not swing between "
        "extremes: alternating 2 and 16 ships between meetings is a policy "
        "failure, not caution. As a rule, change the launching rate by at most "
        "3-4 ships from the current level; large changes only when the treasury "
        "has collapsed or the fleet is far from strength. If debt approaches the "
        "credit limit, cut the launching rate - a company that cannot build or "
        "pay interest dies slowly. Dividends are paid only when they create no "
        "new debt; skipping them too long raises unrest (~+0.2/year, price "
        "collapses above unrest 3).\n"
        'Decide: div_rate (0-20), form ("cash" or "spices"), war (true/false), '
        "ships (1-20, within free capital plus remaining credit).\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            m = re.search(r"\{.*\}", r.json()["message"]["content"], re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [55]:
!cp metrics_llm_council.csv metrics_llm_council_v5_pendulum.csv
!cp council_log.csv council_log_v5_pendulum.csv

import importlib, voc_council, voc_engine_v3
importlib.reload(voc_council); importlib.reload(voc_engine_v3)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes") or [], {k: v for k, v in d.items() if k != "votes"}
    spread = (f"{min(v['div_rate'] for v in votes)}-{max(v['div_rate'] for v in votes)}"
              if votes else "none")
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{int(d2["war"])},'
                f'{d2["max_launch"]},{len(votes)},{spread}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged

from voc_engine_v3 import BASE, MODERATE
open("council_log.csv", "w").write("year,div_rate,form,war,ships,n_votes,div_spread\n")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
r = df[df["year"] == 1660].iloc[0]
print(f"1660: price={r['voc_price']} cum_div={r['cum_div']} debt={r['debt_mln']} "
      f"ships={r['ships']} unrest={r['unrest']} regents={r['regent_share']}")


year,div_rate,form,war,ships,n_votes,div_spread
1602,0.0,spices,1,8,17,0.0-0.0
1607,0.0,cash,1,10,17,0.0-0.0
1612,0.0,cash,1,10,17,0.0-0.0
1617,0.0,cash,1,12,17,0.0-0.0
1622,0.0,spices,1,12,17,0.0-0.0
1627,0.0,spices,1,12,17,0.0-0.0
1632,0.0,cash,1,12,17,0.0-0.0
1637,0.0,cash,1,12,17,0.0-0.0
1642,0.0,cash,1,12,17,0.0-0.0
1647,0.0,cash,1,12,17,0.0-0.0
1652,0.0,spices,1,12,17,0.0-0.0
1657,0.0,spices,1,12,17,0.0-0.0

1660: price=10.0 cum_div=0.0 debt=0.0 ships=12.0 unrest=11.8 regents=72.6


In [56]:
%%writefile voc_council.py
import json, re, statistics
import requests

MODEL = "qwen3:8b"
URL = "http://localhost:11434/api/chat"

HEEREN = ([("Amsterdam", f"merchant-am{i}", 0.5 + 0.1*i) for i in range(8)] +
          [("Zeeland", f"merchant-ze{i}", 0.5 + 0.1*i) for i in range(4)] +
          [("Delft", f"merchant-df{i}", 0.5 + 0.1*i) for i in range(2)] +
          [("Rotterdam", "merchant-rt", 0.5), ("Hoorn", "merchant-hn", 0.6),
           ("Enkhuizen", "merchant-en", 0.7)])

SYSTEM = ("You are a bewindhebber (director) of the VOC, an Amsterdam merchant "
          f"of the chamber of {{city}}, year {{year}}. Historical context: VOC has "
          "permanent capital of 6.42M guilders, trade brings 100-300% on voyages. "
          "Decide the annual policy. Respond ONLY with JSON.")

def ask_one(c, p, t, ctx, attempts=3):
    sys_text = SYSTEM.format(city=c, year=ctx["year"])
    rd = ctx.get("recent_divs")
    divs = "none" if rd in (None, "", []) else (
        ", ".join(f"{d}%" for d in rd) if isinstance(rd, (list, tuple)) else f"{rd}%")
    usr = (
        f"You are bewindhebber {p} of the chamber of {c}.\n"
        f"{ctx['retro']}\n"
        f"Year: {ctx['year']}. VOC treasury: {ctx['cash_mln']:.2f}M guilders. "
        f"Free capital above reserve: {ctx['cash_free_mln']:.2f}M. "
        f"Debt: {ctx['debt_mln']:.2f}M. Share price: {ctx['price']:.0f}.\n"
        f"Recent dividends: {divs}. Years since last dividend: {ctx['years_since_div']}. "
        f"Unrest: {ctx['unrest']:.2f}. War ongoing: {bool(ctx['war'])}.\n"
        f"Fleet at sea: {ctx['fleet_sea']}. Launched last year: {ctx['ships_last']}. "
        f"Revenue last year: {ctx['last_rev_mln']:.2f}M. "
        f"Credit remaining: {ctx['credit_left_mln']:.1f}M "
        f"(one ship ~{ctx['ship_cost_mln']:.2f}M, borrowed money costs 3-5%/yr).\n"
        "Steady-state guidance: a healthy VOC launches a STEADY 8-12 ships per "
        "year and pays a dividend every 5-10 years. Do not swing between "
        "extremes: alternating 2 and 16 ships between meetings is a policy "
        "failure, not caution. As a rule, change the launching rate by at most "
        "3-4 ships from the current level; large changes only when the treasury "
        "has collapsed or the fleet is far from strength. If debt approaches the "
        "credit limit, cut the launching rate - a company that cannot build or "
        "pay interest dies slowly. Dividends are paid only when they create no "
        "new debt; skipping them too long raises unrest (~+0.2/year, price "
        "collapses above unrest 3).\n"
        'Decide: div_rate (0-20), form ("cash" or "spices"), war (true/false), '
        "ships (1-20, within free capital plus remaining credit).\n"
        'Answer ONLY with JSON: {{"div_rate": N, "form": "...", "war": ..., "ships": N}}')
    for i in range(attempts):
        try:
            r = requests.post(URL, json={
                "model": MODEL, "think": False,
                "messages": [{"role": "system", "content": sys_text},
                             {"role": "user", "content": usr}],
                "options": {"temperature": t}, "stream": False}, timeout=120)
            raw = r.json()["message"]["content"]
            with open("council_raw.log", "a") as f:
                f.write(f"=== {ctx['year']} {c} {p} t={t} ===\n{raw[:500]}\n")
            m = re.search(r"\{.*\}", raw, re.S)
            v = json.loads(m.group(0))
            form = "spices" if "spice" in str(v["form"]).lower() else "cash"
            return dict(div_rate=float(min(20, max(0, v["div_rate"]))),
                        form=form, war=bool(v["war"]),
                        ships=int(min(20, max(1, v["ships"]))))
        except Exception as e:
            if i == 0: print("ask_one ошибка:", repr(e))
            continue
    return None

def assemble(ctx):
    votes = [v for v in (ask_one(c, p, t, ctx) for c, p, t in HEEREN) if v]
    if not votes:
        return dict(div_rate=0.0, form="spices", war=False, max_launch=10)
    half = len(votes) / 2
    return {"div_rate": statistics.median(v["div_rate"] for v in votes),
            "form": "spices" if sum(v["form"] == "spices" for v in votes) > half else "cash",
            "war": sum(v["war"] for v in votes) > half,
            "max_launch": statistics.median(v["ships"] for v in votes),
            "votes": votes}


Overwriting voc_council.py


In [57]:
src = open("voc_engine_v3.py").read()

def sub(old, new):
    global src
    assert src.count(old) == 1, f"неуникально или не найдено: {old[:70]!r}"
    src = src.replace(old, new)

sub("""                council_pol = voc_council.assemble(ctx)
                if council_pol["div_rate"] > 0:""",
    """                council_pol = voc_council.assemble(ctx)
                if (council_pol["div_rate"] <= 0
                        and (year - (last_div_year if last_div_year else 1601)) >= 5
                        and (cash - p['div_reserve']) > 0.25 * nominal):
                    council_pol["div_rate"] = 2.5
                    council_pol["form"] = "cash"
                    print(f"{year}: ревизия участников вынуждает выплату 2.5% наличными")
                if council_pol["div_rate"] > 0:""")

open("voc_engine_v3.py", "w").write(src)
print("патч 6 применён")


патч 6 применён


In [58]:
!cp metrics_llm_council.csv metrics_llm_council_v6_steady.csv
!cp council_log.csv council_log_v6_steady.csv

import importlib, voc_council, voc_engine_v3
importlib.reload(voc_council); importlib.reload(voc_engine_v3)

_orig = voc_council.assemble
def assemble_logged(ctx):
    d = _orig(ctx)
    votes, d2 = d.get("votes") or [], {k: v for k, v in d.items() if k != "votes"}
    spread = (f"{min(v['div_rate'] for v in votes)}-{max(v['div_rate'] for v in votes)}"
              if votes else "none")
    with open("council_log.csv", "a") as f:
        f.write(f'{ctx["year"]},{d2["div_rate"]},{d2["form"]},{int(d2["war"])},'
                f'{d2["max_launch"]},{len(votes)},{spread}\n')
    d["votes"] = votes
    return d
voc_council.assemble = assemble_logged

from voc_engine_v3 import BASE, MODERATE
open("council_log.csv", "w").write("year,div_rate,form,war,ships,n_votes,div_spread\n")
open("council_raw.log", "w").write("")
df = voc_engine_v3.run_dutch(dict(BASE, **MODERATE), seed=42)
df.to_csv("metrics_llm_council.csv", index=False)

print(open("council_log.csv").read())
r = df[df["year"] == 1660].iloc[0]
print(f"1660: price={r['voc_price']} cum_div={r['cum_div']} debt={r['debt_mln']} "
      f"ships={r['ships']} unrest={r['unrest']} regents={r['regent_share']}")


1607: ревизия участников вынуждает выплату 2.5% наличными
year,div_rate,form,war,ships,n_votes,div_spread
1602,0.0,cash,1,8,17,0.0-0.0
1607,0.0,cash,1,10,17,0.0-0.0
1612,2.0,spices,1,10,17,2.0-2.5
1617,2.0,spices,1,12,17,2.0-2.0
1622,2.0,spices,1,12,17,2.0-2.0
1627,2.0,spices,1,12,17,2.0-2.0
1632,2.0,spices,1,12,17,2.0-2.0
1637,2.0,spices,1,12,17,2.0-2.0
1642,2.0,cash,1,12,17,2.0-2.0
1647,2.0,spices,1,12,17,2.0-2.0
1652,2.0,spices,1,12,17,2.0-2.0
1657,2.0,cash,1,12,17,2.0-2.0

1660: price=159.9 cum_div=92.1 debt=0.0 ships=12.0 unrest=0.0 regents=63.5


In [59]:
import pandas as pd
base, council = pd.read_csv("metrics_base.csv"), pd.read_csv("metrics_llm_council.csv")
rows = []
for name, d in [("base (правило)", base),
                ("v2 осторожность", pd.read_csv("metrics_llm_council_v2_caution.csv")),
                ("v4 накопительство", pd.read_csv("metrics_llm_council_v4_hoard.csv")),
                ("v6 steady", pd.read_csv("metrics_llm_council_v6_steady.csv")),
                ("v7 совет+ревизия", council)]:
    r = d[d["year"] == 1660].iloc[0]
    rows.append(dict(прогон=name, цена=r["voc_price"], дивиденды=r["cum_div"],
                     долг=r["debt_mln"], unrest=r["unrest"], регенты=r["regent_share"]))
pd.DataFrame(rows).round(1)


,прогон,цена,дивиденды,долг,unrest,регенты
0,base (правило),368.5,413.0,0.0,0.0,46.5
1,v2 осторожность,26.8,25.8,23.3,8.6,68.3
2,v4 накопительство,10.0,0.0,0.0,11.8,72.2
3,v6 steady,10.0,0.0,0.0,11.8,72.6
4,v7 совет+ревизия,159.9,92.1,0.0,0.0,63.5


In [60]:
import os
for f in sorted(os.listdir(".")):
    sz = os.path.getsize(f) / 1024
    print(f"{f:45s} {sz:8.1f} KB")


.config                                            4.0 KB
__pycache__                                        4.0 KB
council_log.csv                                    0.4 KB
council_log_v2_caution.csv                         0.3 KB
council_log_v3_moderate.csv                        0.3 KB
council_log_v4_hoard.csv                           0.3 KB
council_log_v5_pendulum.csv                        0.3 KB
council_log_v6_steady.csv                          0.4 KB
council_raw.log                                   20.1 KB
mc_raw.csv                                       997.0 KB
metrics_base.csv                                   4.7 KB
metrics_cf1.csv                                    4.7 KB
metrics_cf2.csv                                    4.7 KB
metrics_cf3.csv                                    4.6 KB
metrics_cf4.csv                                    4.7 KB
metrics_llm_council.csv                            4.7 KB
metrics_llm_council_v2_caution.csv                 4.6 KB
metrics_llm_co

In [61]:
from google.colab import drive
drive.mount("/content/drive")

import datetime, shutil, os
STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
DST = "/content/drive/MyDrive/voc_sim_2026"

os.makedirs(DST, exist_ok=True)
shutil.make_archive(f"{DST}/voc_sim_{STAMP}", "zip", ".", ".")
print(f"архив: {DST}/voc_sim_{STAMP}.zip, {os.path.getsize(f'{DST}/voc_sim_{STAMP}.zip')/1024/1024:.1f} MB")


Mounted at /content/drive


OSError: [Errno 95] Operation not supported: 'drive/MyDrive/Новый документ (11).gdoc'

In [62]:
import datetime, os, zipfile

STAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M")
DST = "/content/drive/MyDrive/voc_sim_2026"
os.makedirs(DST, exist_ok=True)

SKIP_EXT = {".gdoc", ".gform", ".gsheet", ".gslides", ".gdrawio", ".shortcut"}

candidates = [f for f in os.listdir(".") if os.path.isfile(f)
              and os.path.splitext(f)[1].lower() not in SKIP_EXT]

zpath = f"{DST}/voc_sim_{STAMP}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in candidates:
        z.write(f)
print(f"архив: {zpath}, {os.path.getsize(zpath)/1024/1024:.2f} MB, файлов: {len(candidates)}")


архив: /content/drive/MyDrive/voc_sim_2026/voc_sim_20261114_1504.zip, 1.22 MB, файлов: 25
